In [ ]:
SELECT table_name, column_name
FROM spark_catalog.silver.information_schema.columns
WHERE lower(column_name) LIKE '%service%'
   OR lower(column_name) LIKE '%clienttype%'
   OR lower(column_name) LIKE '%tenancy%'
ORDER BY table_name, column_name;

In [ ]:
SELECT DISTINCT trim(clienttype) AS v
FROM <that_table>
WHERE clienttype IS NOT NULL AND trim(clienttype) <> ''
LIMIT 50;

In [ ]:
# Search columns across all tables in a database (Lakehouse schema)
db = "silver"
patterns = ["service", "clienttype", "tenancy"]

tables = [t.name for t in spark.catalog.listTables(db) if t.tableType.lower() != "view"]
hits = []

for tbl in tables:
    cols = [c.name.lower() for c in spark.table(f"{db}.{tbl}").schema.fields]
    if any(any(p in col for p in patterns) for col in cols):
        for c in cols:
            if any(p in c for p in patterns):
                hits.append((tbl, c))

display(spark.createDataFrame(hits, ["table_name", "column_name"]).orderBy("table_name", "column_name"))

--------------new------------------

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH src AS (
    SELECT DISTINCT
        cp.z_src_system_instance                          AS service_src_sys_inst_id,
        trim(cp.cprod_service)                            AS service_src_name,
        lower(trim(cp.cprod_service))                     AS service_src_name_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'
),
src_with_id AS (
    SELECT
        service_src_sys_inst_id,
        service_src_name,
        concat(service_src_sys_inst_id, '|', service_src_name_norm) AS service_src_id_norm
    FROM src
),
rdm AS (
    SELECT DISTINCT
        lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)
SELECT
    -- 3 mandatory fields (per Mali)
    s.service_src_id_norm     AS service_src_id,
    s.service_src_name        AS service_src_name,
    s.service_src_sys_inst_id AS service_src_sys_inst_id,

    -- useful audit
    current_timestamp()       AS z_src_created_date_time,
    current_user()            AS z_src_created_by_user,
    current_timestamp()       AS z_src_modified_date_time,
    current_user()            AS z_src_modified_by_user,

    1                         AS z_order_is_active
FROM src_with_id s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT count(*) AS new_rows FROM silver.silver_rdm_service_add;

%sql
SELECT service_src_sys_inst_id, count(*) AS c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

%sql
SELECT * FROM silver.silver_rdm_service_add LIMIT 50;